# 02 — Deterministic Feature Extraction

Extract the 5 deterministic features from every sample, validate they correlate with the fatigue labels in the expected direction, then fit and save normalization stats for the CNN training notebook.

In [ ]:
import sys, json
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))

DATA_DIR = ROOT / 'data' / 'synthetic'

import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

from simulate import FS
from features import extract_features, normalize_features

%matplotlib inline
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})

## Load dataset

In [ ]:
X = np.load(DATA_DIR / 'X.npy')
y = np.load(DATA_DIR / 'y.npy')
print(f'X: {X.shape}  y: {y.shape}')

## Extract deterministic features

Runs `extract_features()` on all 1000 samples. Expect ~1–2 minutes (sample entropy is O(n²) on the burst envelope).

In [ ]:
det = np.zeros((len(X), 5), dtype=np.float32)
for i, sig in enumerate(tqdm(X, desc='Extracting features')):
    det[i] = extract_features(sig, FS)

print(f'det shape: {det.shape}')
print(f'Feature ranges:')
feat_names = ['mean_rms', 'mdf_slope', 'spectral_comp', 'rms_trend', 'entropy']
for name, col in zip(feat_names, det.T):
    print(f'  {name:18s}  min={col.min():.3f}  max={col.max():.3f}  mean={col.mean():.3f}')

## Feature vs fatigue label — scatter plots

Each feature plotted against `fatigue_end` (label 0). Expected directions:
- `mean_rms` → increases with fatigue (neural drive)
- `mdf_slope` → more negative with fatigue (spectral shift)
- `spectral_comp` → decreases with fatigue (energy moves to low freq)
- `rms_trend` → increases with fatigue (late bursts louder than early)
- `entropy` → decreases with fatigue (more regular signal)

In [ ]:
fatigue_label = y[:, 0]  # fatigue_end

fig, axes = plt.subplots(1, 5, figsize=(16, 3))
for ax, name, col in zip(axes, feat_names, det.T):
    ax.scatter(fatigue_label, col, s=4, alpha=0.3, color='steelblue')
    # Trend line
    z = np.polyfit(fatigue_label, col, 1)
    xfit = np.linspace(0, 1, 100)
    ax.plot(xfit, np.polyval(z, xfit), 'r-', lw=1.5)
    ax.set_xlabel('fatigue_end')
    ax.set_title(name, fontsize=9)

fig.suptitle('Deterministic features vs fatigue label')
plt.tight_layout()

## Correlation matrix

Check for redundancy between features and labels. High inter-feature correlation means the CNN has less independent information from each branch.

In [ ]:
label_names = ['fatigue_end', 'mean_fatigue', 'mdf_slope_norm', 'spectral_comp_gt', 'entropy_gt']
all_cols = np.hstack([det, y])
all_names = feat_names + label_names

corr = np.corrcoef(all_cols.T)

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)
plt.colorbar(im, ax=ax)
ax.set_xticks(range(len(all_names)))
ax.set_yticks(range(len(all_names)))
ax.set_xticklabels(all_names, rotation=45, ha='right', fontsize=8)
ax.set_yticklabels(all_names, fontsize=8)
for i in range(len(all_names)):
    for j in range(len(all_names)):
        ax.text(j, i, f'{corr[i,j]:.2f}', ha='center', va='center', fontsize=6)
ax.set_title('Pearson correlation — features and labels')
plt.tight_layout()

## Fit normalization stats

Fit on the full dataset here. In training notebook we refit on train split only — this cell is just for inspection.

In [ ]:
det_norm, stats = normalize_features(det)

print('Normalization stats (will be refit on train split in notebook 03):')
for name, mu, sigma in zip(feat_names, stats['mean'], stats['std']):
    print(f'  {name:18s}  mean={mu:.4f}  std={sigma:.4f}')

fig, axes = plt.subplots(1, 5, figsize=(16, 3))
for ax, name, col in zip(axes, feat_names, det_norm.T):
    ax.hist(col, bins=30, color='steelblue', edgecolor='white', lw=0.3)
    ax.set_title(name, fontsize=9)
    ax.set_xlabel('Normalised value')
fig.suptitle('Normalised feature distributions (should be ~N(0,1))')
plt.tight_layout()

## Save outputs

In [ ]:
np.save(DATA_DIR / 'det_features.npy', det)

stats_json = {k: v.tolist() for k, v in stats.items()}
(DATA_DIR / 'feat_stats.json').write_text(json.dumps(stats_json, indent=2))

print(f'Saved det_features.npy and feat_stats.json to {DATA_DIR}')
print('Proceed to 03-cnn-training.ipynb')